# Word Embeddings

Dans ce TP, nous allons voir comment utiliser des word embeddings pré-appris à l'aide de [`spacy`](https://spacy.io/) et [`gensim`](https://radimrehurek.com/gensim/).

Nous verrons ensuite comment apprendre nos propres embeddings.

In [ ]:
%pip install gensim

In [ ]:
import itertools
import os
import pathlib
import sys
import re
import typing

import gensim.models
import gensim.models.phrases
import spacy

## Modèles pré-entrainés avec Spacy

Afin d'utiliser des modèles de langue dans spacy, il faut préalablement les télécharger. Il existe des modèles d'embeddings correspondant à plusieurs langues :

- Anglais
- Allemand
- Grec
- Espagnol
- Français
- Italien
- Lithuanien
- Norvegien
- Hollandais
- Portugais

Une quarantaine d'autre langues sont en prévision. (cf. [modèles de langues spacy](https://spacy.io/usage/models))

Nous allons travailler avec le modèle le plus abouti, celui de l'anglais comportant des vecteurs d'embeddings pour 20 000 mots.

In [ ]:
!python -m spacy download en_core_web_sm
#!python -m spacy download fr_core_news_md
#!python -m spacy download de
# …

## Utilisation
Après avoir chargé le modèle, on peut traiter des documents comme suit :

In [ ]:
# Chargement du modèle
nlp = spacy.load('en_core_web_sm')

# Traitement du document avec Spacy
doc = nlp("This is some text that I am processing with Spacy")

print("Le vecteur dense correspondant au 4ème mot")
print(doc[3].vector)

# Le vecteur dense moyen de la phrase. Utilisation à des fin de classification
# par exemple (même si dans l'absolu c'est une mauvaise idée : il vaudrait mieux
# utiliser les librairies fasttext ou transformers)
print("vecteur moyen")
print(doc.vector)

## Modèles pré-entrainés avec Gensim

Gensim ne vient pas avec ses propres vecteurs denses d'embeddings de mots. On peut cependant aisément en utiliser, pour peu qu'on les télécharge.

De nombreuses sources de modèles pré-entrainés d'embeddings sont disponibles en téléchargement.

- Le plus populaire est sans doute le Google News dataset model à 300 dimensions (cf. [Google News word2vec documentation](https://code.google.com/archive/p/word2vec/); [GoogleNews-vectors-negative300.bin.gz](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing))

- Une source de modèles variés pour beaucoup de langues : [NLPL word embeddings repository](http://vectors.nlpl.eu/repository/)

- Un modèle développé par Facebook qui apprend des embeddings de parties de mots [fasttext](https://fasttext.cc/docs/en/crawl-vectors.html)

- Un dépôt pour le français : [Jean-Philippe Fauconnier](https://fauconnier.github.io/)

Dans la suite on utilisera le modèle français du dépôt de Jean-Philippe Fauconnier (continous bag of word, dimension 200, cutoff 100, ni lemmatisation, ni pos-tagging, ni-phrasing).

In [ ]:
!wget https://embeddings.net/embeddings/frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin

In [ ]:
!ls -l
model_file = "frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin"

## Utilisation de [`gensim.models.KeyedVectors`](https://radimrehurek.com/gensim/models/keyedvectors.html)

In [ ]:
# Chargement d'un modèle de word embeddings
model = gensim.models.KeyedVectors.load_word2vec_format(model_file, binary=True)
print("nombre de mots dans le dictionnnaire : ",len(model.index_to_key))

In [ ]:
for i, word in enumerate(model.index_to_key):
  print(word)
  if i > 50:
    break

In [ ]:
# Transformer un mot en vecteur
vector = model['facile']

# Transformer un phrase en liste de vecteur.
# Il faut que les entrées soient tokenizées.
# Si un mot n'est pas le vocabulaire, vous aurez une erreur.
# Il faut être attentif aux prétraitements que l'on utilise par rapport au
# modèle que l'on utilise (lemmatisation, lowercasing, etc).
words = "Ceci est une phrase transformée à l'aide de Gensim".lower().split(' ')
vectors = [(word, model[word]) for word in words if word in model.key_to_index]
print(f"Mots transformés : {' '.join(word for word, _ in vectors)}")

In [ ]:
model.most_similar('berlin')

In [ ]:
print(model.similarity('berlin', 'munich'))
print(model.similarity('berlin', 'paris'))

## Opération dans l'espace sémantique
À l'aide de la fonction [`gensim.models.keyedvectors.KeyedVectors.similar_by_vector`](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.similar_by_vector) définissez une fonction qui effectue l'opération de translation sémantique vue en cours.

Cette fonction prend 3 arguments `x`, `y` et `z` et répond à la question suivante :

`x` est à `y` ce que `z` est à ?

Ex : *Roi* est à *Reine* ce que *Oncle* est à ?

Réponse : *Tante*

Il est conseillé d'afficher plusieurs résultats, c'est plus simple à coder et on trouve plus facilement ce que l'on cherche (Il arrive souvent que la première réponse soit `z`…)

In [ ]:
def semantic_analogy(x, y, z):
  # Votre code ici
  return "Tante"

### Solution

In [ ]:
def semantic_analogy(x, y, z):
  return model.similar_by_vector(model[y] - model[x] + model[z])

## À vous de jouer
Utilisez cette fonction sur des exemples qui vous viennent.

In [ ]:
semantic_analogy("homme", "femme", "chien")

## Création de Word Embeddings

L'apprentissage de représentations denses sur vos propres données n'est pas compliqué. Cela peut être particulièrement adapté quand on travaille sur des corpus de documents spécifiques (acte notariés, langues anciennes, …)

### Le corpus 20 Newsgroups

Ce corpus est constitué d'environ 20 000 « posts » séparés en 20 thématiques (car provenant de 20 newsgroups différents).

In [ ]:
!git clone https://github.com/nzmonzmp/20Newsgroups.git
!tar xzf 20Newsgroups/20news-bydate.tar.gz
!ls 20news-bydate-train/

In [ ]:
!ls 20news-bydate-test/

In [ ]:
# Chaque dossier correspond à un newsgroup, puis un article par fichier
contents = [p.read_text(encoding="latin-1")
            for p in pathlib.Path(".").glob("20news-bydate-*/*/*")]


print(f"{len(contents)} textes récupérés")

In [ ]:
print(contents[1])

In [ ]:
# Nettoyage des ponctuations et tokenization.
def preprocess(content: str) -> typing.List[str]:
  offset = content.find('\n\n')
  if offset > 0:
    content = content[offset + 2:]
  sentences = (re.sub(r'[\!"#$%&\*+,-./:;<=>?@^_`()|~=]', " ", line).split()
               for line in content.splitlines()
               if not line.endswith("writes:"))
  return list(itertools.chain.from_iterable(sentences))


texts = [preprocess(content) for content in contents]

In [ ]:
print(texts[0])

## Détection de groupe nominaux comun en utilisant Gensim Phraser

Certains bigrammes de mots sont très communs (Ex : New York). Il est souvent opportun de détecter ces bigrammes afin de les traiter comme des "mots" du dictionnaire.

C'est l'utilité de l'objet Phraser qui va permettre de détecter et transformer en conséquence notre corpus.

In [ ]:
# Création des groupes nominaux courants
phrases = gensim.models.phrases.Phrases(
    texts, connector_words=gensim.models.phrases.ENGLISH_CONNECTOR_WORDS)

# L'objet Phraser va maintenant nous permettre de transformer notre corpus
phraser = gensim.models.phrases.Phraser(phrases)
phrased_texts = list(phraser[texts])
len(phrased_texts)

In [ ]:
trainable_model = gensim.models.Word2Vec(
    phrased_texts,
    min_count=3,      # Seuil à partir duquel les mots ne sont pas ignorés
    vector_size=200,  # Dimension des word embeddings
    workers=2,        # Nombre de threads
    window=5,         # Taille de la fenêtre de contexte
    epochs=30)        # Nombre d'itérations
model = trainable_model.wv  # On récupère l'instance KeyedVectors après
                            # l'entraînement

In [ ]:
model.most_similar("New_York")

In [ ]:
semantic_analogy("wind", "fly", "water")